## Read cleaned data

In [8]:
data = "/home/slark/DE-projects/Commercial-data/super_store/data/02_clean/01_cleaned_store_data.parquet"

In [4]:
%%bash 

pip install pandas pyarrow

In [ ]:
import pandas as pd
import pyarrow as pa

In [9]:
df = pd.read_parquet(data, engine="pyarrow")

In [10]:
df.head()

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
0,AG-2011-2040,2011-01-01,2011-01-06,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,Office Supplies,Storage,"Tenex Lockers, Blue",408.0,2,0.0,106.140,35.46,Medium,2011
1,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Office Supplies,Supplies,"Acme Trimmer, High Speed",120.0,3,0.1,36.036,9.72,Medium,2011
2,HU-2011-1220,2011-01-01,2011-01-05,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,...,Office Supplies,Storage,"Tenex Box, Single Width",66.0,4,0.0,29.640,8.17,High,2011
3,IT-2011-3647632,2011-01-01,2011-01-05,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,Office Supplies,Paper,"Enermax Note Cards, Premium",45.0,3,0.5,-26.055,4.82,High,2011
4,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",114.0,5,0.1,37.770,4.70,Medium,2011


In [11]:
df.dtypes

order_id                  object
order_date        datetime64[ns]
ship_date         datetime64[ns]
ship_mode                 object
customer_name             object
segment                   object
state                     object
country                   object
market                    object
region                    object
product_id                object
category                  object
sub_category              object
product_name              object
sales                    float64
quantity                   int64
discount                 float64
profit                   float64
shipping_cost            float64
order_priority            object
year                       int64
dtype: object

In [ ]:
# df = pd.read_csv(
#     data, 
#     parse_dates=['order_date', 'ship_date'],
#     dayfirst=False
#     )

#Date format lost through converting to csv then back to df. 
#Using parse_dates to keep date format. 

In [ ]:
#df.dtypes

order_id                     str
order_date        datetime64[us]
ship_date         datetime64[us]
ship_mode                    str
customer_name                str
segment                      str
state                        str
country                      str
market                       str
region                       str
product_id                   str
category                     str
sub_category                 str
product_name                 str
sales                    float64
quantity                   int64
discount                 float64
profit                   float64
shipping_cost            float64
order_priority               str
year                       int64
dtype: object

# Fact table 

## Add columns

In [50]:
# days from order to shipping 

df['order_prep_days'] = df['ship_date'] - df['order_date']

In [51]:
# profit margin

df['profit_margin_%'] = round((df['profit'] / df['sales']) * 100 , 2)

In [54]:
df['GM2_%'] = round((((df['profit'] - df['shipping_cost']) / df['sales']) * 100), 2)

In [57]:
# loss making orders

df['loss_making'] = df['profit'] < 1 

In [ ]:
# discounted orders / discount bins (0-10%, 10-20% etc.)

def discount_bin(row): 
    if row['discount'] == 0:
        return 'no discount'
    elif row['discount'] < 0.11:
        return '1-10%'
    elif row['discount'] < 0.21: 
        return '11-20%'
    elif row['discount'] < 0.31: 
        return '21-30%'
    elif row['discount'] < 0.41: 
        return '31-40%'
    elif row['discount'] < 0.51: 
        return '41-50%'
    elif row['discount'] < 0.61: 
        return '51-60%'
    elif row['discount'] < 0.71: 
        return '61-70%'
    elif row['discount'] < 0.81: 
        return '71-80%'
    elif row['discount'] < 0.91: 
        return '81-90%'
    elif row['discount'] < 1: 
        return '91-100%'
    
df['discount_range'] = df.apply(lambda row: discount_bin(row), axis=1) # check this out further 

In [60]:
df.head()

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,profit,shipping_cost,order_priority,year,order_prep_days,GM2,profit_margin_%,GM2_%,loss_making,discount_range
0,AG-2011-2040,2011-01-01,2011-01-06,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,106.140,35.46,Medium,2011,5 days,17.32,26.01,17.32,False,no discount
1,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,36.036,9.72,Medium,2011,7 days,21.93,30.03,21.93,False,1-10%
2,HU-2011-1220,2011-01-01,2011-01-05,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,...,29.640,8.17,High,2011,4 days,32.53,44.91,32.53,False,no discount
3,IT-2011-3647632,2011-01-01,2011-01-05,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,-26.055,4.82,High,2011,4 days,-68.61,-57.90,-68.61,True,41-50%
4,IN-2011-47883,2011-01-01,2011-01-08,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,37.770,4.70,Medium,2011,7 days,29.01,33.13,29.01,False,1-10%


In [ ]:
#sales value banding 

def categorise_order_value(row): 
    if row['sales'] == 0:
        return 'no revenue'
    elif row['sales'] < 101:
        return 'very small'
    elif row['sales'] < 501:
        return 'small'
    elif row['sales'] < 5001 : 
        return 'medium'
    else: 
        return 'large'
   
    
df['order_value_range'] = df.apply(lambda row: categorise_order_value(row), axis=1) 

#may need to review the groupings here


In [ ]:
df.to_csv('02_column_check.csv')

#TODO: save to SQL

Data ready to be loaded to SQL 

Does shipping days have SLA relation to priority group? 
